In [1]:
!pip install easyocr pytesseract jiwer python-Levenshtein diff-match-patch Pillow pandas PyMuPDF numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 32.6 MB/s eta 0:00:00


In [9]:
!sudo apt-get update
!sudo apt-get install -y tesseract-ocr-spa

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,946 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu

In [11]:
import os
import re
import unicodedata
import logging
from pathlib import Path
from dataclasses import dataclass

# OCR & Data libraries
import easyocr
import pytesseract
from PIL import Image
import pandas as pd
import numpy as np
import fitz

# 1.Adapted Evaluator Class & Data Models

In [12]:
logger = logging.getLogger(__name__)

@dataclass
class EvaluationReport:
    page_id: str
    cer_score: float
    wer_score: float
    frequent_errors: dict
    error_heatmap_path: str = None
    char_diff_html: str = None

class Evaluator:
    _LIGATURES = {
        "ﬁ": "fi", "ﬂ": "fl", "ﬃ": "ffi", "ﬄ": "ffl",
        "ﬀ": "ff", "ﬅ": "st", "ﬆ": "st",
    }

    def __init__(self, output_dir: str | Path = "data/output_images/eval", normalise: bool = True) -> None:
        self.output_dir = Path(output_dir)
        self.normalise = normalise
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def evaluate(self, page_id: str, ocr_text: str, gt_text: str, image_path: str | Path | None = None) -> EvaluationReport:
        if self.normalise:
            ocr_text = self._normalise_text(ocr_text)
            gt_text = self._normalise_text(gt_text)

        cer = self.compute_cer(ocr_text, gt_text)
        wer = self.compute_wer(ocr_text, gt_text)

        frequent_errors = self.build_confusion_matrix(ocr_text, gt_text)
        char_diff_html = self.generate_semantic_diff_html(ocr_text, gt_text)

        return EvaluationReport(
            page_id=page_id,
            cer_score=cer,
            wer_score=wer,
            frequent_errors=frequent_errors,
            error_heatmap_path=None,
            char_diff_html=char_diff_html,
        )

    def compute_cer(self, hypothesis: str, reference: str) -> float:
        from jiwer import cer as jiwer_cer
        ref_norm = self._normalise_text(reference)
        hyp_norm = self._normalise_text(hypothesis)
        if not ref_norm and not hyp_norm: return 0.0
        if not ref_norm: return 100.0
        return float(jiwer_cer(ref_norm, hyp_norm)) * 100.0

    def compute_wer(self, hypothesis: str, reference: str) -> float:
        from jiwer import wer as jiwer_wer
        ref_norm = self._normalise_text(reference)
        hyp_norm = self._normalise_text(hypothesis)
        if not ref_norm and not hyp_norm: return 0.0
        if not ref_norm: return 100.0
        return float(jiwer_wer(ref_norm, hyp_norm)) * 100.0

    def build_confusion_matrix(self, hypothesis: str, reference: str) -> dict[str, dict[str, int]]:
        import Levenshtein
        matrix: dict[str, dict[str, int]] = {}
        for op, hyp_pos, ref_pos in Levenshtein.editops(hypothesis, reference):
            if op == "replace":
                gt_char = reference[ref_pos]
                pred_char = hypothesis[hyp_pos]
                if gt_char not in matrix:
                    matrix[gt_char] = {}
                matrix[gt_char][pred_char] = matrix[gt_char].get(pred_char, 0) + 1
        return {
            gt_char: dict(sorted(preds.items(), key=lambda x: x[1], reverse=True))
            for gt_char, preds in sorted(matrix.items())
        }

    @staticmethod
    def _normalise_text(text: str) -> str:
        if not text: return ""
        for ligature, expansion in Evaluator._LIGATURES.items():
            text = text.replace(ligature, expansion)
        text = re.sub(r"\^[a-zA-Záéíóú]{1,2}", "", text)
        text = text.lower()
        nfd = unicodedata.normalize("NFD", text)
        out, i = [], 0
        while i < len(nfd):
            c = nfd[i]
            if c == "n" and i + 1 < len(nfd) and nfd[i + 1] == "\u0303":
                out.extend([c, "\u0303"])
                i += 2
            elif unicodedata.category(c) == "Mn":
                i += 1
            else:
                out.append(c)
                i += 1
        text = unicodedata.normalize("NFC", "".join(out))
        text = re.sub(r"[^\w\s]", "", text, flags=re.UNICODE)
        normalized = re.sub(r"\s+", " ", text)
        return normalized.strip()

    def generate_semantic_diff_html(self, hypothesis: str, reference: str) -> str:
        from diff_match_patch import diff_match_patch
        ref_norm = self._normalise_text(reference)
        hyp_norm = self._normalise_text(hypothesis)
        dmp = diff_match_patch()
        dmp.Diff_Timeout = 0.0
        diffs = dmp.diff_main(ref_norm, hyp_norm)
        html_output = []
        for op, data in diffs:
            text = data.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
            if op == dmp.DIFF_INSERT:
                html_output.append(f'<span style="background-color: #d4edda; color: #155724;">{text}</span>')
            elif op == dmp.DIFF_DELETE:
                html_output.append(f'<span style="background-color: #f8d7da; color: #721c24; text-decoration: line-through;">{text}</span>')
            elif op == dmp.DIFF_EQUAL:
                html_output.append(f"<span>{text}</span>")
        return "".join(html_output)

# 2. Image Processing & Evaluation Pipeline

In [13]:
def resize_image_if_needed(img: Image.Image, max_dim: int = 2048) -> Image.Image:
    """Resizes a PIL Image so its longest edge is at most `max_dim` pixels."""
    w, h = img.size
    if max(w, h) > max_dim:
        # Calculate ratio to maintain aspect
        ratio = max_dim / float(max(w, h))
        new_size = (int(w * ratio), int(h * ratio))

        # LANCZOS is the high-quality downsampling filter
        img = img.resize(new_size, Image.Resampling.LANCZOS)
        print(f"  -> Resized image from {w}x{h} to {new_size[0]}x{new_size[1]}")
    return img

def process_and_evaluate(pdf_dir, gt_dir):
    evaluator = Evaluator(normalise=True)

    # 1. EASYOCR UPDATE: Change language to Spanish
    print("Loading EasyOCR Spanish model...")
    reader = easyocr.Reader(['es'])

    results = []
    pdf_files = sorted([f for f in os.listdir(pdf_dir) if f.lower().endswith('.pdf')])

    for pdf_file in pdf_files[:5]:
        base_name = os.path.splitext(pdf_file)[0]
        pdf_path = os.path.join(pdf_dir, pdf_file)
        gt_path = os.path.join(gt_dir, f"{base_name}.txt")

        if not os.path.exists(gt_path):
            print(f"Ground truth missing for {pdf_file}. Skipping.")
            continue

        print(f"\nProcessing: {pdf_file}...")

        with open(gt_path, 'r', encoding='utf-8') as f:
            gt_text = f.read().strip()

        doc = fitz.open(pdf_path)
        page = doc.load_page(0)
        pix = page.get_pixmap(dpi=300)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        img = resize_image_if_needed(img, max_dim=2048)

        # 2. EASYOCR UPDATE: Add contrast adjustments for handwritten strokes
        img_np = np.array(img)
        easyocr_results = reader.readtext(
            img_np,
            detail=0,
            paragraph=True,
            adjust_contrast=0.5, # Boosts contrast for faded ink
            text_threshold=0.6   # Slightly lower threshold to catch thin strokes
        )
        easyocr_text = " ".join(easyocr_results)

        # 3. TESSERACT UPDATE: Set Spanish language, LSTM engine, and layout analysis
        # --oem 1 uses the LSTM neural net, --psm 6 assumes a uniform block of text
        tess_config = r'--oem 1 --psm 6'
        tesseract_text = pytesseract.image_to_string(
            img,
            lang='spa',
            config=tess_config
        )

        easyocr_report = evaluator.evaluate(
            page_id=f"{base_name}_easyocr", ocr_text=easyocr_text, gt_text=gt_text
        )

        tesseract_report = evaluator.evaluate(
            page_id=f"{base_name}_tesseract", ocr_text=tesseract_text, gt_text=gt_text
        )

        results.append({
            "Document": pdf_file,
            "Engine": "EasyOCR",
            "CER (%)": round(easyocr_report.cer_score, 2),
            "WER (%)": round(easyocr_report.wer_score, 2)
        })
        results.append({
            "Document": pdf_file,
            "Engine": "Tesseract",
            "CER (%)": round(tesseract_report.cer_score, 2),
            "WER (%)": round(tesseract_report.wer_score, 2)
        })

    return pd.DataFrame(results)

# 3. Run the Pipeline

In [14]:
# Ensure these point to the folders containing your PDFs and TXT files
PDF_DIRECTORY = "/content/drive/MyDrive/GSoC-26-Eval-Data/pdfs"
GROUND_TRUTH_DIRECTORY = "/content/drive/MyDrive/GSoC-26-Eval-Data/ground_truth"

os.makedirs(PDF_DIRECTORY, exist_ok=True)
os.makedirs(GROUND_TRUTH_DIRECTORY, exist_ok=True)

# Run the pipeline
df_results = process_and_evaluate(PDF_DIRECTORY, GROUND_TRUTH_DIRECTORY)

# Display the final table
display(df_results)

Loading EasyOCR Spanish model...

Processing: ahpg-gpah 1 1716a 35 - 1744.pdf...
  -> Resized image from 12600x16800 to 1536x2048

Processing: ahpg-gpah au61 2 - 1606.pdf...
  -> Resized image from 12600x16800 to 1536x2048

Processing: es 28079 ahn inquisición1667exp 12 - 1640.pdf...
  -> Resized image from 2657x3728 to 1459x2048

Processing: pleito entre el marqués de viana.pdf...
  -> Resized image from 2614x3708 to 1443x2047

Processing: pt3279 146 342 - 1857_page_1.pdf...
  -> Resized image from 12600x16800 to 1536x2048


,Document,Engine,CER (%),WER (%)
0,ahpg-gpah 1 1716a 35 - 1744.pdf,EasyOCR,56.98,98.94
1,ahpg-gpah 1 1716a 35 - 1744.pdf,Tesseract,60.21,119.58
2,ahpg-gpah au61 2 - 1606.pdf,EasyOCR,44.23,93.33
3,ahpg-gpah au61 2 - 1606.pdf,Tesseract,65.32,109.17
4,es 28079 ahn inquisición1667exp 12 - 1640.pdf,EasyOCR,68.51,98.94
5,es 28079 ahn inquisición1667exp 12 - 1640.pdf,Tesseract,66.70,95.24
6,pleito entre el marqués de viana.pdf,EasyOCR,72.05,114.03
7,pleito entre el marqués de viana.pdf,Tesseract,51.73,100.90
8,pt3279 146 342 - 1857_page_1.pdf,EasyOCR,62.24,97.40
9,pt3279 146 342 - 1857_page_1.pdf,Tesseract,63.04,95.83
